In [1]:
from pathlib import Path

import cantera
import polars
import util
from util.sim import reactors

tag = util.notebook_prefix()

In [2]:
DATA_PATH = Path("../data")

In [3]:
# Read in data and rename species to match simulation
species_df = polars.read_csv(DATA_PATH / "webb" / "species.csv")
species_dct = dict(species_df["concentration", "name"].drop_nulls().iter_rows())
conc_df = polars.read_csv(DATA_PATH / "webb" / "concentration.csv")
conc_df = conc_df.rename(species_dct, strict=False)

In [4]:
model = cantera.Solution(DATA_PATH / "cantera" / f"full_{tag}_calc.yaml")
concs = conc_df.select("CPT(563)", "N2", "O2(6)").rows(named=True)

In [5]:
temp = 825
pres = 1.1 * cantera.one_atm  # in atm.
tau = 4  # s
vol = 1 * (1e-2) ** 3  # m3

In [6]:
# Run simulations for each point and store the results in an array
solns = cantera.SolutionArray(model)
for conc in concs:
    print(f"Starting simulation for {conc}")
    reactor = reactors.jsr(model=model, temp=temp, pres=pres, tau=tau, vol=vol, conc=conc)
    solns.append(reactor.thermo.state)

Starting simulation for {'CPT(563)': 0.005, 'N2': 0.9825, 'O2(6)': 0.0125}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.97625, 'O2(6)': 0.01875}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.973571429, 'O2(6)': 0.021428571}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.97, 'O2(6)': 0.025}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.965, 'O2(6)': 0.03}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.960909091, 'O2(6)': 0.034090909}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.9575, 'O2(6)': 0.0375}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.953333333, 'O2(6)': 0.041666667}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.945, 'O2(6)': 0.05}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.937307692, 'O2(6)': 0.057692308}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.9325, 'O2(6)': 0.0625}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.926818182, 'O2(6)': 0.068181818}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.92, '

In [7]:
sim_df = conc_df.with_columns(
    *(polars.Series(s, solns(s).X.flatten() * 10**6) for s in species_df["name"])
)
sim_df.write_csv(DATA_PATH / "cantera" / f"full_{tag}_calc.csv")
sim_df

phi,O2_molecules,CPT(563),N2,O2(6),H2(2),H2O(5),CO(12),CO2(13),CH4(33),CH3CHO(41),C2H4(52),C3H6(131),C3H4O(165),C4H6(227),C4H8(253),C5H6(478),C5H8(522),C5H8O(825)rs,C6H6(970),HO2(8),OH(4)
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
3.0,0.12233,2301.524021,978503.70709,8647.73946,737.875995,3192.949789,2246.103049,255.720432,273.024751,37.571506,1564.971428,124.550438,357.008762,21.568123,43.234763,57.455685,577.86768,17.536116,2.12289,1.93368,0.001012
2.0,0.1835,1507.969198,970308.07609,12082.30686,953.937461,5327.810915,4198.183176,604.594805,380.589015,50.325926,1881.644614,115.298087,333.137571,22.682507,34.699328,55.062156,578.382138,17.816176,2.849615,3.540389,0.002097
1.75,0.20962,1362.169209,967273.885821,14017.973408,969.245034,5896.874482,4639.867708,724.785206,382.524399,50.062512,1885.873175,105.462177,313.562988,22.005314,29.92024,55.311771,581.299456,18.016004,2.774499,4.074798,0.002442
1.5,0.24456,1247.479955,963498.520119,16924.051249,950.569153,6414.693073,4952.87736,849.238131,369.359189,47.605161,1848.845894,93.456093,289.545114,21.316401,24.826779,57.174102,594.130939,18.606333,2.606424,4.636088,0.002774
1.25,0.29347,1163.980052,958489.205653,21387.999991,891.958187,6845.072622,5085.948586,965.834376,341.171692,43.055942,1768.239041,80.151412,261.526442,20.830824,19.812789,61.388536,620.971046,19.752826,2.372454,5.200672,0.003059
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.17,2.15785,1039.50309,771171.530144,211572.876434,210.047456,7405.051401,3119.202041,708.206323,72.5262,10.488632,588.680801,15.338579,87.889076,18.067347,2.083454,136.881926,941.817779,33.246178,0.694179,7.384873,0.00358
0.15,2.44557,1038.198572,741981.161865,240973.730285,189.583825,7383.301409,3014.81684,665.500644,64.095325,9.680039,536.050427,13.922451,83.250527,17.841237,1.809653,140.124978,949.928993,33.598516,0.628414,7.426817,0.003586
0.13,2.75147,1036.846078,703776.940935,279407.570932,168.430027,7356.319128,2901.590738,618.791662,55.334961,8.835337,480.520232,12.483849,78.418262,17.596871,1.537685,143.444605,957.537888,33.94413,0.557262,7.471368,0.003592
